# EECE 5644 Mini Project 1

In [413]:
############################################################

# EECE 5644 - Machine Learning
# Skylar Denno
# June 30, 2026

# MINI PROJECT 1
# Data cleaning and data preprocessing

############################################################

In [414]:
import sys, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.2f}".format)
np.random.seed(0)

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

In [415]:
### 3.2 Getting Information

# load raw dataset from csv
raw_df = pd.read_csv("data.csv", encoding="ISO-8859-1")
print("\nLoaded dataset with", raw_df.shape[0], "rows")

# class
class GettingInfo:
    def __init__(self):
        pass

    def print_basic_description(self, df):
        print("\n\n--------------------\nBASIC INFORMATION\n--------------------")
        print("\nShape:", df.shape)
        print("\nInfo:", df.info())
        print("\nStatistical Description:", df.describe())
        print("\nHead (first 10 rows):", df.head(10))

basic_info = GettingInfo()
basic_info.print_basic_description(raw_df)


Loaded dataset with 541909 rows


--------------------
BASIC INFORMATION
--------------------

Shape: (541909, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB

Info: None

Statistical Description:        Quantity  UnitPrice  CustomerID
count 541909.00  541909.00   406829.00
mean       9.55       4.61    15287.69
std      218.08      96.76     1713.60
min   -80995.00  -11062.06    12346.00
25%        1.00       1.25    13953.00
50%       

In [416]:
### 3.9 Unique Values

class UniqueValues:
    def __init__(self):
        pass

    def print_unique_values(self, df):
        print("\n\n--------------------\nUNIQUE VALUES/COUNTS\n--------------------")
        print("\nNum. of countries: ", df["Country"].nunique())
        print("\nNum. of instances of each country present: ", df["Country"].value_counts())

        print("\nNum. of stock codes: ", df["StockCode"].nunique())
        print("\nNum. of stock codes (INCLU. MISSING): ", df["StockCode"].nunique(dropna=False))
        print("\nNum. of instances of each stock code: ", df["StockCode"].value_counts())
        print("\nNum. of instances of each stock code (INCLU. MISSING):")
        print(df["StockCode"].value_counts(dropna=False))

        print("\nNum. of unique product descriptions:", df["Description"].nunique())
        print("\nNum. of instances of each product description", df["Description"].value_counts())

unique_values = UniqueValues()
unique_values.print_unique_values(raw_df)



--------------------
UNIQUE VALUES/COUNTS
--------------------

Num. of countries:  38

Num. of instances of each country present:  Country
United Kingdom          495478
Germany                   9495
France                    8557
EIRE                      8196
Spain                     2533
Netherlands               2371
Belgium                   2069
Switzerland               2002
Portugal                  1519
Australia                 1259
Norway                    1086
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
Unspecified                446
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
Israel                     297
USA                        291
Hong Kong                  288
Singapore                  229
Iceland                    182
Canada                     151
Greece               

In [417]:
### 3.8 Min/max/sum/mean/count

class NumericSummary:
    def __init__(self):
        pass

    def print_numeric_summary(self, df):
        print("\n\n--------------------\nNUMERIC SUMMARY\n--------------------")
        summary = df[["Quantity", "UnitPrice"]].agg(["min", "max", "sum", "mean", "count"])
        print(summary)

numeric_summary = NumericSummary()
numeric_summary.print_numeric_summary(raw_df)



--------------------
NUMERIC SUMMARY
--------------------
        Quantity  UnitPrice
min    -80995.00  -11062.06
max     80995.00   38970.00
sum   5176450.00 2498803.97
mean        9.55       4.61
count  541909.00  541909.00


In [418]:
### 3.1 Creating a DataFrame

class CountryRegionLookup:
    def __init__(self):
        self.country_region_lookup = None

    def build_country_region_lookup(self):
        country_region_data = [
            ("United Kingdom", "UK & IE"),
            ("Germany", "Western Europe"),
            ("France", "Western Europe"),
            ("Spain", "Western Europe"),
            ("Netherlands", "Western Europe"),
            ("Belgium", "Western Europe"),
            ("Switzerland", "Western Europe"),
            ("Portugal", "Western Europe"),
            ("Australia", "Oceania"),
            ("Norway", "Northern Europe"),
            ("Italy", "Southern Europe"),
            ("Channel Islands", "Western Europe"),
            ("Finland", "Northern Europe"),
            ("Cyprus", "Mediterranean"),
            ("Sweden", "Northern Europe"),
            ("Austria", "Western Europe"),
            ("Denmark", "Northern Europe"),
            ("Japan", "Asia Pacific"),
            ("Poland", "Central Europe"),
            ("Israel", "Middle East"),
            ("USA", "North America"),
            ("Hong Kong", "Asia Pacific"),
            ("Singapore", "Asia Pacific"),
            ("Iceland", "Northern Europe"),
            ("Canada", "North America"),
            ("Greece", "Southern Europe"),
            ("Malta", "Southern Europe"),
            ("United Arab Emirates", "Middle East"),
            ("Lebanon", "Middle East"),
            ("Lithuania", "Eastern Europe"),
            ("Brazil", "South America"),
            ("Czech Republic", "Central Europe"),
            ("Bahrain", "Middle East"),
            ("Saudi Arabia", "Middle East"),
            ("RSA", "Africa"),
            ("ERIE", "Ireland"),
            ("European Community", "GENERAL EUROPE"),
            ("Unspecified", "UNSPECIFIED")
        ]

        country_region_dict = dict(country_region_data)
        country_region_dict.setdefault("DEFAULT", "INVALID COUNTRY")
        self.country_region_lookup = country_region_dict
        return self.country_region_lookup

country_lookup = CountryRegionLookup()
country_region_lookup_table = country_lookup.build_country_region_lookup()

In [419]:
# start of cleaning -> make a copy of raw dataframe to clean
clean_df = raw_df.copy()
clean_df = clean_df.dropna(subset=["CustomerID", "Description"])

### 3.3 Slicing
class SliceRows:
    def __init__(self):
        pass

    def slice_rows(self, df):
        df = df.copy()
        df.index = [f"item_{i}" for i in range(len(df))]

        print("\n\n--------------------\nSLICING WITH ILOC AND LOC\n--------------------")
        print("\nFirst 5 rows with iloc:")
        print(df.iloc[0:5])

        labels = [f"item_{i}" for i in range(5)]
        print("\nRows retrieved by label with loc:")
        print(df.loc[labels])
        return df

slicer = SliceRows()
clean_df = slicer.slice_rows(clean_df)



--------------------
SLICING WITH ILOC AND LOC
--------------------

First 5 rows with iloc:
       InvoiceNo StockCode                          Description  Quantity  \
item_0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
item_1    536365     71053                  WHITE METAL LANTERN         6   
item_2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
item_3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
item_4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

           InvoiceDate  UnitPrice  CustomerID         Country  
item_0  12/1/2010 8:26       2.55    17850.00  United Kingdom  
item_1  12/1/2010 8:26       3.39    17850.00  United Kingdom  
item_2  12/1/2010 8:26       2.75    17850.00  United Kingdom  
item_3  12/1/2010 8:26       3.39    17850.00  United Kingdom  
item_4  12/1/2010 8:26       3.39    17850.00  United Kingdom  

Rows retrieved by label with loc:
       InvoiceNo StockC

In [420]:
### 3.4 Conditional Selection
# THIS ALSO DELETES THOSE ROWS

class FilterCanceled:
    def __init__(self):
        pass

    def remove_canceled_orders(self, df):
        df = df.copy()
        df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
        return df

canceled_remover = FilterCanceled()
clean_df = canceled_remover.remove_canceled_orders(clean_df)

In [421]:
### 3.5 Sorting Values && 3.18 Applying a Function
# NOTE: THIS CLASS ALSO ADDS THE REVENUE COLUMN TO THE DATAFRAME.

class LargestOrders:
    def __init__(self):
        pass

    def find_largest_orders(self, df, top_n=5):
        df = df.copy()

        #########
        df["revenue"] = df["Quantity"] * df["UnitPrice"]
        #########

        print("\n\n--------------------\nTOP BULK ORDERS BY QUANTITY\n--------------------")
        print(df.sort_values("Quantity", ascending=False).head(top_n)[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "revenue"]])

        print("\n\n--------------------\nTOP ORDERS BY REVENUE\n--------------------")
        print(df.sort_values("revenue", ascending=False).head(top_n)[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "revenue"]])

        print("\n\n--------------------\nTOP RETURNS BY NEGATIVE QUANTITY\n--------------------")
        returns = df[df["Quantity"] < 0]
        print(returns.sort_values(["Quantity", "revenue"], ascending=[True, True]).head(top_n)[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "revenue"]])

        return df.sort_values(["Quantity", "revenue"], ascending=[False, False])

largest_orders = LargestOrders()
clean_df = largest_orders.find_largest_orders(clean_df)



--------------------
TOP BULK ORDERS BY QUANTITY
--------------------
            InvoiceNo StockCode                        Description  Quantity  \
item_406349    581483     23843        PAPER CRAFT , LITTLE BIRDIE     80995   
item_38120     541431     23166     MEDIUM CERAMIC TOP STORAGE JAR     74215   
item_378894    578841     84826     ASSTD DESIGN 3D PAPER STICKERS     12540   
item_315541    573008     84077  WORLD WAR 2 GLIDERS ASSTD DESIGNS      4800   
item_146796    554868     22197               SMALL POPCORN HOLDER      4300   

             UnitPrice   revenue  
item_406349       2.08 168469.60  
item_38120        1.04  77183.60  
item_378894       0.00      0.00  
item_315541       0.21   1008.00  
item_146796       0.72   3096.00  


--------------------
TOP ORDERS BY REVENUE
--------------------
            InvoiceNo StockCode                          Description  \
item_406349    581483     23843          PAPER CRAFT , LITTLE BIRDIE   
item_38120     541431     2

In [422]:
### 3.6 Replacing Values

class ReplaceVals:
    def __init__(self):
        pass

    def replace_values(self, df):
        df = df.copy()
        df["Country"] = df["Country"].replace({
            "EIRE": "Ireland",
            "RSA": "South Africa",
            "Unspecified": "missing"
        })
        return df

clean_df = ReplaceVals().replace_values(clean_df)

In [423]:
### 3.7 Renaming Columns

class RenameColumns:
    def __init__(self):
        pass

    def rename_columns(self, df):
        df = df.copy()
        df = df.rename(columns={
            "InvoiceNo": "invoice_no",
            "StockCode": "stock_code",
            "Description": "description",
            "Quantity": "quantity",
            "InvoiceDate": "invoice_date",
            "UnitPrice": "unit_price",
            "CustomerID": "customer_id",
            "Country": "country"
        })
        return df

clean_df = RenameColumns().rename_columns(clean_df)

In [424]:
### 3.10 Handle Missing Values

# My reasoning: Using a chi-square test, we check if missingness is associated with other variables
# or if it is not very associated with other values and therefore likely just by random chance.
# We use a standard cutoff of alpha=0.05 to determine if missingness is related to other variables.
# p-value > 0.05 --> Assume MCAR and discard this row
# KEEP MAR AND MNAR. DISCARD MCAR

from scipy.stats import ttest_ind

class RemoveMCAR:
    def __init__(self):
        pass

    def findMCAR(self, df):
        df = df.copy()
        target_cols = [col for col in ["description", "customer_id"] if col in df.columns]

        for col in target_cols:
            if df[col].isna().sum() == 0:
                continue

            missing_indicator = df[col].isna().astype(int)
            p_values = []

            for feature in ["quantity", "unit_price"]:
                if feature in df.columns:
                    present = df.loc[missing_indicator == 0, feature].dropna()
                    missing = df.loc[missing_indicator == 1, feature].dropna()
                    if len(present) > 1 and len(missing) > 1:
                        _, p = ttest_ind(present, missing, equal_var=False)
                        p_values.append((feature, p))

            if "country" in df.columns:
                country_present = df.loc[missing_indicator == 0, "country"]
                country_missing = df.loc[missing_indicator == 1, "country"]
                if len(country_present) > 1 and len(country_missing) > 1:
                    from scipy.stats import chi2_contingency
                    crosstab = pd.crosstab(missing_indicator, df["country"])
                    _, p, _, _ = chi2_contingency(crosstab)
                    p_values.append(("country", p))

            if p_values:
                p_value_summary = ", ".join(f"{feature}={p:.4f}" for feature, p in p_values)
                print(f"{col}: p-values -> {p_value_summary}")
                max_p = max(p for _, p in p_values)
                print(f"{col}: highest p-value = {max_p:.4f}")
                if max_p > 0.05:
                    print(f"{col}: MCAR trigger -> rows with missing {col} will be removed")
                    df = df.dropna(subset=[col])

        return df

### TESTS DESIGNED TO GIVE P VALUES JUST ABOVE 0.05
np.random.seed(0)

# missing discriptions
test_df_1 = pd.DataFrame({
    "description": ["a", "b", None, None, "c", "d", None, None],
    "quantity": [1, 2, 1.1, 2.1, 3, 4, 3.2, 4.1],
    "unit_price": [10, 11, 10.2, 11.1, 12, 13, 12.1, 13.2],
    "country": ["UK", "UK", "US", "US", "UK", "UK", "US", "US"]
})

# missing customer_ids
test_df_2 = pd.DataFrame({
    "customer_id": [101, 102, None, None, 105, 106, None, None],
    "quantity": [2, 3, 2.1, 3.1, 4, 5, 4.1, 5.1],
    "unit_price": [20, 21, 20.2, 21.1, 22, 23, 22.1, 23.2],
    "country": ["US", "US", "UK", "UK", "US", "US", "UK", "UK"]
})

print("Test dataframe 1 result:")
print(RemoveMCAR().findMCAR(test_df_1))
print("\nTest dataframe 2 result:")
print(RemoveMCAR().findMCAR(test_df_2))
### END TESTS

clean_df = RemoveMCAR().findMCAR(clean_df)

Test dataframe 1 result:
description: p-values -> quantity=0.8961, unit_price=0.8749, country=0.0339
description: highest p-value = 0.8961
description: MCAR trigger -> rows with missing description will be removed
  description  quantity  unit_price country
0           a      1.00       10.00      UK
1           b      2.00       11.00      UK
4           c      3.00       12.00      UK
5           d      4.00       13.00      UK

Test dataframe 2 result:
customer_id: p-values -> quantity=0.9163, unit_price=0.8749, country=0.0339
customer_id: highest p-value = 0.9163
customer_id: MCAR trigger -> rows with missing customer_id will be removed
   customer_id  quantity  unit_price country
0       101.00      2.00       20.00      US
1       102.00      3.00       21.00      US
4       105.00      4.00       22.00      US
5       106.00      5.00       23.00      US


In [425]:
### 3.11 Delete Column

# My reasoning: invoice number is not relevant to my analysis, and it doesn't really mean anything in the "trends"
# the business questions are asking about. They are for the company's internal use to track orders, and
# they are unrelated to who the customer actually is; they could be completely random as far as we are concerned.
# We have already deleted the canceled orders so it has no more useful data.

# I thought initially description could be removed, reasoning that it was redundant with stock code.
# However, we see with nunique above that there are more unique descriptions than stock codes, so they are not
# exchangable or 1-to-1. Therefore, there is some more complex relationship between stock code and description
# so both are kept.

class DeleteColumn:
    def __init__(self):
        pass

    def drop_column(self, df, column_name):
        df = df.copy()
        if column_name in df.columns:
            df = df.drop(columns=[column_name])
            print(f"Dropped column: {column_name}")
        else:
            print(f"Column not found: {column_name}")
        return df

clean_df = DeleteColumn().drop_column(clean_df, "invoice_no")

Dropped column: invoice_no


In [426]:
### 3.12 Delete Row

In [427]:
cleaned_info = GettingInfo()
cleaned_vals = UniqueValues()
cleaned_info.print_basic_description(clean_df)



--------------------
BASIC INFORMATION
--------------------

Shape: (397924, 8)
<class 'pandas.core.frame.DataFrame'>
Index: 397924 entries, item_406349 to item_366848
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   stock_code    397924 non-null  object 
 1   description   397924 non-null  object 
 2   quantity      397924 non-null  int64  
 3   invoice_date  397924 non-null  object 
 4   unit_price    397924 non-null  float64
 5   customer_id   397924 non-null  float64
 6   country       397924 non-null  object 
 7   revenue       397924 non-null  float64
dtypes: float64(3), int64(1), object(4)
memory usage: 27.3+ MB

Info: None

Statistical Description:        quantity  unit_price  customer_id   revenue
count 397924.00   397924.00    397924.00 397924.00
mean      13.02        3.12     15294.32     22.39
std      180.42       22.10      1713.17    309.06
min        1.00        0.00     12346.00      0.00
25